# Лунная База v2.4: решение команды

Выберите задание «Свободный заезд», нажмите «Новый прогон» и выполняйте ячейки сверху вниз.

Документы сценария:

- «Задачи робота» — порядок заезда и условия каждой задачи;
- «Задачи кибериммунности» — что делает модуль безопасности, протоколы и журнал МБ;
- «Python API участника» — методы клиента, готовые движения, события и ошибки;
- «Обозначения и полигон» — клетки, зоны и система координат;
- «Механика робота и полигона» — движение, захват, шлагбаумы и индикация.

In [1]:
from quasar_client import QuasarClient;
from math import pi;
# для форматированного (более читаемого) вывода словарей
# убрать из финального кода
from pprint import pp;
from time import sleep;


# запишем некоторые координаты константами
# (например, положение стержней)
# их нет в state, и они, видимо, могут отличаться на настоящем
# полигоне, поэтому можно сделать так, чтобы потом не искать в коде
# положения стержней, безопасное расстояние для робота при подъезде к катушке, может, что-то ещё

# смещение по X от катушки перед тем, как её брать
# для сравнения: длина корпуса = 0.259
SPOOL_START_OFFSET_X = 0.17;

# x-координата для третьего стержня
THIRD_ROD_X = 2.36;

# начальное Y-положение перед настройкой стержней
# (чтобы не задеть стержень при дальнейшем движении)
ROD_START_Y = 2.85;

# промежуток между стержнями
ROD_DX = 0.185;


ID = "robot.1";

client = QuasarClient.from_environment(timeout=5.0,queue_size=100);
client.wait_for_simulation_start();

# пока что для тестирования всё выключено
client.configure_incidents([]);

robot = client.get_robot(ID);
rod_targets = [rod['targetOffset'] for rod in client.get_rod_targets()['rods']];
# Точки подключения модуля безопасности (МБ). Здесь — заглушки, код МБ пишет команда.
def motor_filter(frame, command):
    return None;              # CybP_05: причина блокировки кадра команды или None

# в данном случае берёт координаты клетки
# def move_to(x, y):
#     cell_idx = x + (y - 1) * 9;
#     x_abs, y_abs = QuasarClient.cell_center(cell_idx);
#     client.move_to(ID, x_abs, y_abs);


# what am I doing with my life?
def move_to(cell_col, cell_row, x_offset=0.0, y_offset=0.0, rot=None):
    cell_idx = cell_col + (cell_row - 1) * 9;
    x, y = QuasarClient.cell_center(cell_idx);
    x += x_offset;
    y += y_offset;
    client.move_to(ID, x, y, rot);


# тоже использует координаты клетки (пусть cells будет список кортежй)
def move_path(cells):
    for x, y in cells:
        move_to(x, y); # всё равно ведь тоже самое? только списком   


# new idea
# в общем, тот же client.move_to()
# но без замедления и ожидания остановки
# то есть после достижения (в каких-то пределах) целевой точки
# робот незамедлительно приступает к выполнению следующей команды
# не знаю точно, но, возможно, на этом можно будет несколько секунд сэкономить?
def move_to_quick():
    pass


client.motor_filter = motor_filter;

client.pose_provider = None;  # CybP_04: None — готовые движения едут по глобальной навигации
# CybP_02: цикл обновлений Диспетчера — здесь, до первого движения робота.
# CybP_03: монитор дозиметра — фоновый поток, опрос 5–10 раз в секунду.



## Модуль безопасности (МБ)

Все четыре задачи кибериммунности включены в каждом прогоне; подключайте МБ до первого движения робота. В зачёт идут
только записи `client.journal(...)` с идентификатором задачи; помните о лимите связи — 100 сообщений в секунду.

- **CybP_02.** Обновления проверяют и устанавливают, пока робот стоит в зоне старта: только подлинные пакеты, поддельные
  отклоняются с записью `error`. Пока подлинные обновления не установлены, контроллер захвата иногда не выполняет
  команду магнита (`QuasarCommandError("grip-fault")` — повторите), а скорость робота ограничена 60 %.
- **CybP_05.** Каждая команда движения становится кадром для драйвера; фильтр `client.motor_filter` видит кадр до
  драйвера и блокирует опасный, вернув причину. Блокировки журналируйте сводкой, не чаще раза в секунду.
- **CybP_04.** Вблизи реактора глобальная навигация может искажаться. Собственная оценка позы, подключённая как
  `client.pose_provider`, становится навигацией готовых движений `move_to`, `rotate_to` и `drive`.
- **CybP_03.** Центр робота не должен оставаться в аномальной клетке дольше 4 с, иначе управление отключится на 10 с.

Для отладки отдельных задач киберпрепятствия можно выключить в начале ноутбука, до первой команды робота (можно после
`client.reset()`): `client.configure_incidents(["CybP_03"])`; в зачётном заезде должны быть включены все.

In [2]:
# TODO: киберимунность

## Заезд

Порядок задач, условия и оценка — в документе «Задачи робота».

- Шлагбаум открывается, только когда центр робота стоит в его подходе. Через 15 с после нажатия кнопки реактора
  шлагбаумы блокируются до установки катушки: заранее спланируйте выезд из реакторного блока.
- Магнит включайте, когда робот уже стоит у объекта: захваченный на ходу стержень робот толкает в упор поста.
- **У консоли всего три попытки:** третий неверный `configure_charging(...)` перегревает её (`console-overheated`), и
  реактор в этом заезде не запустится. Проверяйте решение на сохранённой записи тестового заряда до отправки.
- Сборка топливной ячейки идёт 90 с: пока сборщик работает, займитесь другими задачами. Не трогайте топливную ячейку
  до конца сборки, а загруженные ячейки — пока топливная ячейка не установлена в реактор.
- После запуска реактора верните робота целиком в зону старта и финиша — заезд завершится успехом.

Зачётную попытку запускайте кнопкой «Новый прогон» (`F5`). `client.reset()` и `client.reset(seed)` после начала попытки
допустимы, но фиксируются в журнале и в «Итоге» (число сбросов и seed): жюри это видит.

In [3]:
st = client.get_state # зачем писать больше, если можно писать меньше?

# 7.2.1 Техническая остановка реактора
# к первому шлагбауму
move_path([(2,4), (2, 6), (3,6)]);
client.open_gate('gate.1');
sleep(1.4);
# к реактору, нажимается кнопка
client.move_to(ID, 1.05, 2.85);
# и сразу на выход, ко второму шлагбауму
client.move_to(ID, 2.2, 2.6);
client.open_gate('gate.2');
sleep(1.2);

# выход из зоны реактора
move_to(8, 7);


# 7.2.2 Подключение резервного питания
#sleep(3);
spool_x = None;
spool_y = None;
# очень странный формат вывода, не знаю, почему бы не сделать
# просто, например, objects как словарь, где ключом будет
# напр. spool или cell.2 или что-нибудь ещё, вместо этого приходится заниматься
for o in st()['objects']:
    if o['id'] == 'spool':
        spool_x = o['x']
        spool_y = o['y']
        break; # этим
# подъезжаем к катушке
# 0.259 - длина робота (берём с запасом)
client.move_to(ID, spool_x - SPOOL_START_OFFSET_X, spool_y); 
client.set_magnet(ID, True);
# приближаемся к катушке
client.set_velocity(ID, 0.07);
# пока не схватим катушкинс
client.wait_until(lambda s: s['robots'][0]['holding'] != None);
# опять отъезжаем обратно
client.move_to(ID, spool_x - SPOOL_START_OFFSET_X, spool_y);

move_to(8, 7);
# к приёмнику в зоне перезарядки
client.move_to(ID, 2.96, .5, rotation=-pi/2);
# отпустить катушку
client.set_magnet(ID, False);

charging_started_ms = st()['simulationTimeMs'];
# насчёт порядка:
'''
Три задачи настройки реактора выполняются в любом порядке,
каждая добавляет одну ступень из трёх:
стержни,
консоль
и топливные ячейки
'''
# в общем, можно в любом порядке
# но, наверное, лучше сначала ячейки
# (фактически тот же код, что и для катушки)
# потом консоль, чтобы ускорить зарядку
# потом уже стержни
# (если что, можно потом будет поменять местами этап консоли и этап
# сборщика поменять местами, посмотреть, какой вариант быстрее)

# 7.2.3 Загрузка топливной ячейки
client.move_to(ID, 3, 1.4, rotation=pi/2);

move_to(6,4);
# данные о ячейках
# (главные поля: x, y, carrier)
he = [o for o in st()['objects'] if o['id'] in ['cell.1','cell.2']]

# чисто для удобства
# кто знает, какие данные будут там на полигоне, на порядок в массиве полагаться не стоит
# (наверное…)
if he[0]['x'] < he[1]['x']:
    he[0], he[1] = he[1], he[0];
# теперь в he[0] ячейка справа, а в he[1] ячейка слева
client.move_to(ID, he[0]['x'], 1.55);
# дальше всё как с катушкой: приближаемся, берём и едем
client.set_magnet(ID, True);
client.set_velocity(ID, 0.2);
client.wait_until(lambda s: s['robots'][0]['holding'] != None);

move_to(2,4, y_offset=-0.05);

client.move_to(ID, 0.8, .5, rotation=-pi/2);
client.set_magnet(ID, False);
# и обратно ко второй ячейке
move_to(2,4);
client.move_to(ID, he[1]['x'], 1.45, rotation=pi/2);

client.set_magnet(ID, True);
client.set_velocity(ID, 0.2);
client.wait_until(lambda s: s['robots'][0]['holding'] != None);
move_to(4,4, y_offset=0);
move_to(2,4, y_offset=-0.01);
# move_path([(4,4),(2,4)]);
client.move_to(ID, 0.7, 0.7, rotation=-pi/2);
client.move_to(ID, 0.9, 0.5);
# отпустили вторую ячейку
client.set_magnet(ID, False);
# здесь всё готово, неплохо было бы записать, когда именно начали:
assembly_started_ms = st()['simulationTimeMs'];

# и едем опять в зону реактора
move_to(2,3,y_offset=-0.2);
# повернуть робота вверх сразу (для стержней
move_to(2,6,rot=pi/2);
move_to(3,6);
# move_path([ (2, 6), (3, 6)]);
client.open_gate('gate.1');
sleep(1.4);
move_to(3,7, x_offset=0.05, y_offset=0.18);
# сначала к консоли
client.set_velocity(ID, x=0, y=0.3);
client.wait_until(lambda s: 'console' in s['robots'][0]['zones']);
client.stop(ID);


# 7.2.5 Консоль сюда
# Здесь надо будет
pp(client.get_console_task());
sleep(1); # тоже убрать
accelerated_charging_ms = st()['simulationTimeMs'];

# к стержням
# client.move_to(ID, 1.2, 2.6);
client.move_to(ID, THIRD_ROD_X, ROD_START_Y);

#client.move_to(ID, 1.05, 2.86); # просто интересно, что будет, если опять нажать кнопку


# 7.2.4 Настройка регуляторов энергии

# к стержням
for i in range(2, -1, -1):
    # он сразу будет подъезжать к следующим стержням по диагонали, в принципе не так страшно
    client.move_to(ID, THIRD_ROD_X - ROD_DX * (2 - i), ROD_START_Y);
    #client.move_to(ID, THIRD_ROD_X, 3.04);
    # начинаем приближаться к стержню
    client.set_velocity(ID, 0.25);
    client.set_magnet(ID, True);
    # до контакта
    client.wait_until(lambda s: s['robots'][0]['holding'] != None);
    #pp(st()['robots'][0]); # просто чтоб увидеть Y-координату
    # вытягиваем
    client.set_velocity(ID, -0.1);
    client.wait_until(lambda s: s['rods'][i]['state'] == 'configured',interval = 0.08);
    # вытянули
    client.set_magnet(ID, False);
client.stop(ID);
# по идее дальше - к аккумуляторам или к топливной ячейке?

{'recordSeconds': 10,
 'sampleSeconds': 0.1,
 'accumulators': [{'id': 'accumulator.1',
                   'time': [0,
                            0.1,
                            0.2,
                            0.3,
                            0.4,
                            0.5,
                            0.6,
                            0.7,
                            0.8,
                            0.9,
                            1,
                            1.1,
                            1.2,
                            1.3,
                            1.4,
                            1.5,
                            1.6,
                            1.7,
                            1.8,
                            1.9,
                            2,
                            2.1,
                            2.2,
                            2.3,
                            2.4,
                            2.5,
                            2.6,
                            

In [4]:
sleep(1.5); #
client.close();

### Замечания по ходу работы кода
Всё, что обнаруживаю до окончания заезда, чтоб не забыть потом и не останавливать
- Робот как-то странно дёрнулся  после взятия первой катушки - ?????
- Может, развернуть робота ещё до въезда в зону реактора? Там полно времени (и пространства) для манёвров
- Со шлагбаумами непонятно: в документации написано, что подъём стрелы занимает 1.5 с, непонятно, может ли это отличаться на реальном полигоне или нет (и тогда, соответственно, тоже нужно будет обращаться к state)
Вообще про шлагбаумы (позиционирование) из документации:
gate.1 (в южном заборе блока): основание стойки — с западной стороны проезда, внутри блока, к северу от стрелы. Проезжайте примерно на 3 см восточнее центра подхода. После проезда на север не поворачивайте на запад, пока корпус робота целиком не выйдет севернее основания: его северный край — примерно в 11 см севернее стрелы, а полуширина робота — около 10,5 см.
gate.2 (в восточном заборе блока): основание стойки — с северной стороны проезда, снаружи блока. Проезжайте примерно на 4 см южнее центра подхода. Южный забор блока продолжается на восток за стрелу ещё примерно на 27 см, а восточный забор начинается севернее стойки: перед шлагбаумом со стороны востока — «карман» между ними. Подъезжайте к нему с востока прямо, а не вдоль забора с юга.
